In [ ]:
# Step 1: Import libraries
from google.colab import drive
import pandas as pd
import numpy as np
import datetime
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler



In [ ]:
# Step 2: Load dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/weather_data_only.csv'
df = pd.read_csv(file_path)

print("Dataset loaded. Shape:", df.shape)
df.head()

Dataset loaded. Shape: (189888, 6)


,Timestamp,Temperature (°C),Humidity (%),Wind Speed (m/s),Rainfall (mm),Solar Irradiance (W/m²)
0,1/1/2020 0:00,28.993428,75.011269,1.053861,4.140513,185.892561
1,1/1/2020 0:15,27.723471,77.024015,1.085152,9.446997,281.782650
2,1/1/2020 0:30,29.295377,74.732958,3.363800,4.265813,328.942058
3,1/1/2020 0:45,31.046060,87.615995,2.539148,1.038103,336.407064
4,1/1/2020 1:00,27.531693,79.709858,1.366819,4.201393,205.494256


In [ ]:
# Step 3: Data Preprocessing
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

df['hour'] = df['Timestamp'].dt.hour
df['day'] = df['Timestamp'].dt.day
df['month'] = df['Timestamp'].dt.month
df['dayofweek'] = df['Timestamp'].dt.dayofweek
df['year'] = df['Timestamp'].dt.year

feature_cols = ['hour', 'day', 'month', 'dayofweek', 'year']

target_cols = ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (m/s)',
               'Rainfall (mm)', 'Solar Irradiance (W/m²)']

X = df[feature_cols].values
y = df[target_cols].values

In [ ]:
# Step 4: Scaling
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

In [ ]:
# Step 5: Create Time Series Data
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 10
X_seq, y_seq = create_sequences(X_scaled, y_scaled, time_steps)

In [ ]:
# Step 6: Split Train & Test
split_idx = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (151902, 10, 5)
Test : (37976, 10, 5)


In [ ]:
# Step 7: Build LSTM Model
model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(time_steps, X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(len(target_cols))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,165 (78.77 KB)

 Trainable params: 20,165 (78.77 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Step 8: Train Model
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

Epoch 1/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - loss: 0.0127
Epoch 2/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - loss: 0.0123
Epoch 3/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - loss: 0.0123
Epoch 4/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 36s 8ms/step - loss: 0.0122
Epoch 5/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - loss: 0.0122
Epoch 6/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - loss: 0.0122
Epoch 7/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - loss: 0.0122
Epoch 8/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 38s 8ms/step - loss: 0.0122
Epoch 9/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 36s 8ms/step - loss: 0.0122
Epoch 10/10
4747/4747 ━━━━━━━━━━━━━━━━━━━━ 38s 8ms/step - loss: 0.0122


In [ ]:
# Step 9: Evaluation
# Predictions
y_pred = model.predict(X_test)

# Inverse scaling
y_test_inv = scaler_y.inverse_transform(y_test)
y_pred_inv = scaler_y.inverse_transform(y_pred)

# -------------------------------
# REGRESSION METRICS
# -------------------------------
mae = mean_absolute_error(y_test_inv, y_pred_inv)
mse = mean_squared_error(y_test_inv, y_pred_inv)
rmse = mse ** 0.5
r2 = r2_score(y_test_inv, y_pred_inv)

print("\n--- Regression Metrics ---")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

# -------------------------------
# CONVERT TO BINARY
# -------------------------------
y_test_bin = (y_test_inv > y_test_inv.mean(axis=0)).astype(int)
y_pred_bin = (y_pred_inv > y_test_inv.mean(axis=0)).astype(int)

y_test_flat = y_test_bin.ravel()
y_pred_flat = y_pred_bin.ravel()

# -------------------------------
# CLASSIFICATION METRICS
# -------------------------------
print("\n--- Classification Metrics (Converted) ---")

accuracy = accuracy_score(y_test_flat, y_pred_flat)
precision = precision_score(y_test_flat, y_pred_flat, average='macro', zero_division=0)
recall = recall_score(y_test_flat, y_pred_flat, average='macro', zero_division=0)
f1 = f1_score(y_test_flat, y_pred_flat, average='macro', zero_division=0)

roc_auc = roc_auc_score(y_test_flat, y_pred_inv.ravel())

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)

1187/1187 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step

--- Regression Metrics ---
MAE: 10.012645580172222
RMSE: 22.63119939164085
R2 Score: -0.0015861304788063358

--- Classification Metrics (Converted) ---
Accuracy : 0.48373709711396673
Precision: 0.4960281748574028
Recall   : 0.49712106268416767
F1 Score : 0.45281274295719365
ROC AUC  : 0.5214213337609702


In [ ]:
#Step 10: Save Model for Later Use
# Save LSTM model (use .h5 or .keras, NOT joblib)
model.save('weather_lstm_model.h5')

# Save scalers also (IMPORTANT)
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

print("\nLSTM model saved as 'weather_lstm_model.h5'")


LSTM model saved as 'weather_lstm_model.h5'


In [ ]:
#Step 11: Prediction Function
from tensorflow.keras.models import load_model

# Load model and scalers (if needed later)
model = load_model('weather_lstm_model.h5', compile=False)
scaler_X = joblib.load('scaler_X.pkl')
scaler_y = joblib.load('scaler_y.pkl')

def predict_weather(timestamp_str):
    """
    Input: '2020-06-15 12:30'
    Returns: predicted weather values
    """
    dt = pd.to_datetime(timestamp_str)

    # Create input features
    features = np.array([[dt.hour, dt.day, dt.month, dt.dayofweek, dt.year]])

    # Scale input
    features_scaled = scaler_X.transform(features)

    # LSTM expects sequence → repeat same input
    time_steps = 10
    features_seq = np.repeat(features_scaled, time_steps, axis=0)
    features_seq = features_seq.reshape(1, time_steps, features_scaled.shape[1])

    # Predict
    pred_scaled = model.predict(features_seq)

    # Inverse scale
    pred = scaler_y.inverse_transform(pred_scaled)[0]

    result = dict(zip(target_cols, pred))
    return result

In [ ]:
#Step 11: Example Prediction
print("\n--- Example Prediction for '2020-07-01 14:00' ---")

pred_example = predict_weather('2020-07-01 14:00')

for k, v in pred_example.items():
    print(f"{k}: {v:.2f}")


--- Example Prediction for '2020-07-01 14:00' ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
Temperature (°C): 28.11
Humidity (%): 80.31
Wind Speed (m/s): 1.96
Rainfall (mm): 5.01
Solar Irradiance (W/m²): 250.83


In [ ]:
#Step 11: Interactive Prediction
user_input = input("\nEnter a date and time (e.g., 2020-08-15 10:30): ")

try:
    pred_user = predict_weather(user_input)
    print("\nPredicted weather:")

    for k, v in pred_user.items():
        print(f"{k}: {v:.2f}")

except Exception as e:
    print("Error:", e)


Enter a date and time (e.g., 2020-08-15 10:30): 2027-10-12   8:12
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step

Predicted weather:
Temperature (°C): 28.10
Humidity (%): 80.27
Wind Speed (m/s): 1.96
Rainfall (mm): 5.00
Solar Irradiance (W/m²): 250.60
